# Introduction
<img src="https://drive.google.com/uc?export=view&id=1oc0_0mlN0VTZEPMQTg_hnYAC87Lb58MI">

**Welcome to MoSeq2 Extract Modeling Notebook!**

MoSeq2 is a software toolkit for unsupervised modeling and characterization of animal behavior. Moseq takes depth recordings of animals as input and outputs a rich description of behavior as a series of kinematic motifs called 'syllables'. __The model output can be analyzed using the [MoSeq2 Analysis Visualization Notebook](./MoSeq2-Analysis-Visualization-Notebook.ipynb).__

In this notebook, the **Markdown** above each cell describes the purpose of the cell(s), cell output, and the instructions for running the cell(s) and/or interacting with the widget. The **inline code comments** in the code block provides contextual information about the function, code structure, and parameters.

For more detailed documentation, please visit our [Wiki](https://github.com/dattalab/moseq2-app/wiki).
For general feedback and feature requests, please fill out [this survey](https://forms.gle/FbtEN8E382y8jF3p6).

***

## Check MoSeq Versions
- **Run the following cell** to check if `moseq2-app` is installed in your current environment.
- Disregard any warning text with a red background in the output of this cell. They are generated from other packages. They do not mean there is a MoSeq2-related error and do not impact performance or usability for the MoSeq2 pipeline.

This cell should print the version numbers of each MoSeq2 package installed on your system, as well as the path to the python executable that MoSeq2 is run on.

In [ ]:
!moseq2-extract --version
!moseq2-pca --version
!moseq2-model --version
!moseq2-viz --version

import sys
import moseq2_app

print('MoSeq2 app version:', moseq2_app.__version__)
print('Python path:', sys.executable)

# Project setup

## Files and Directory Structure

The currently accepted depth data extensions are:
- `.dat` (raw depth files from our kinect2 data acquisition software)
- `.tar.gz` (compressed depth files from our kinect2 data acquisition software)
- `.avi` (compressed depth files from the `moseq2-extract` CLI)
- `.mkv` (depth recordings from the Azure Kinect and Orbbec Femto acquisition software, under any file name)


**We recommend recording more than 10 hours of depth video (~1 million frames at 30 frames per second) to ensure quality MoSeq models.**

Each MoSeq project should be organized within one project directory.
To better organize the extraction, modeling, and analysis results, you should copy the MoSeq notebooks to the data directory that contains the depth recording to be analyzed (i.e. in  `<data_dir>` in the directory structure below).

At this stage, the data directory should contain **separate subfolders** for each depth recording session, as shown below:

```
.                            ** current working directory
└── <data_dir>/              ** the directory with all depth recordings
    ├── session_1/           ** the folder containing all of a single session's data
    ├   ├── depth.dat        ** the depth data recording itself
    ├   ├── depth_ts.txt     ** csv/txt file of the frame timestamps
    ├   └── metadata.json    ** json file that contains metadata of the rodent (group, subjectName, etc.)
    ...
    ├── session_n/
    ├   ├── depth.dat
    ├   ├── depth_ts.txt
    └── └── metadata.json
```
You can find more information about the file structure [here](https://github.com/dattalab/moseq2-app/wiki/Directory-Structures-and-yaml-Files-in-MoSeq-Pipeline).

## Set up the Project

**[IMPORTANT] ALWAYS run this cell when you open this notebook.** If you don't, the pipeline won't work properly.

The project is the folder holding the depth recordings and, once the next cell writes it, `config.yaml`. Each step of the pipeline records the files it writes in the project's `moseq2-project.yaml`, and the next step reads them from there, so a step run on the command line in the project folder is picked up by the next cell here.

The following cell names the project folder and shows what it holds: how many recordings are extracted, and the sessions, PCA results and models `moseq2-project.yaml` lists.

**Instructions:**
- **Specify the data directory, the folder with all the depth recordings,** in the `base_dir` field. No change is needed if the notebooks are in the base directory.
- **Run the following cell**.

In [ ]:
from moseq2_app.project import start

base_dir = './' # Add the path to your data folder here.

project = start(base_dir)


# Extraction and PCA

Extract the recordings, check the extractions and compute their PCA scores and changepoints in the [MoSeq2 Pipeline Notebook](./MoSeq2-Pipeline.ipynb), steps 1 to 6, run in this project folder. Then come back here to assign groups and fit the model.


## Assign Groups
Sessions can be given group labels for analyses comparing different cohorts or experimental conditions and the labels will be stored in the `groups` section of the project's [config.yaml file](https://github.com/dattalab/moseq2-app/wiki/Directory-Structures-and-yaml-Files-in-MoSeq-Pipeline#configyaml). You can find more information about this widget in the [wiki](https://github.com/dattalab/moseq2-app/wiki/MoSeq2-Extract-Modeling-Notebook-Instructions#assign-groups).

The following cell runs the widget to assign groups and save them in `config.yaml`.

**Instructions:**
- **Run the following cell**.
- **Click the column header** to sort the column and use the filter icon to filter if needed.
- **Click on the session** to select the session. **To select multiple sessions, click the sessions while holding down the [Ctrl]/[Command] key, or click the first and last entry while holding down the [Shift] key.**
- **Enter the group name in the `Desired Group Name` field** and click `Set Group` to update the `group` column for the selected sessions.
- To leave a session out of every step after extraction, tick its `exclude` box; `Save Groups` saves it in the `exclude` list of `config.yaml`.
- Click the `Save Groups` button to save current group assignments.

In [ ]:
from moseq2_app.main import interactive_group_setting

interactive_group_setting(str(project.folder))

# AR-HMM Modeling

## Fitting the AR-HMM
Fitting the AR-HMM typically requires adjusting the `kappa` hyperparameter to achieve a target syllable duration (higher values of `kappa` lead to longer syllable durations). The target mean duration can be determined using changepoints analysis or set heuristically to 0.5-0.6 seconds based on prior literature.

In the code below, set `kappa` to `'scan'` to generate a bash script to run a series of models with different kappa values. You can specify minimum kappa value using `--min-kappa`(eg. 10,000), and maximum kappa value using `max-kappa` (eg. 10,000,000). The bash script will be generated in the specified model directory. You can read more about kappa scan in the wiki [here](https://github.com/dattalab/moseq2-app/wiki/Analysis-Tips#recommended-practice-for-modeling).


You can use the "Get Best Model Fit" cell to pick an optimal value. We recommend fitting for 100-200 iterations to pick `kappa`. For final model fitting, set `kappa` to the chosen value and fit for ~1000 iterations.

You can find more information about AR-HMM modeling parameters in the wiki [here](https://github.com/dattalab/moseq2-app/wiki/MoSeq2-Extract-Modeling-Notebook-Instructions#most-relevant-model-paramters) and learn more about different AR-HMM models we support in the [CLI wiki](https://github.com/dattalab/moseq2-app/wiki/Command-Line-Interface-for-Extraction-and-Modeling#ar-hmm-modeling).

**Note:** if loading a model checkpoint, ensure the modeling parameters (especially the selected groups) are identical to that of the checkpoint. Otherwise, the model will fail.

### Set Model Parameters

Read the inline code comments for more information about the model parameters.

In [ ]:
from moseq2_app.util import update_config

model_name = 'model.p' # the model's file name, saved in the project's models folder

# update the model section of the configuration file
with update_config(project.config_file, section='model') as config_data:
    # model saving frequency (in iterations); will create a checkpoints/ directory containing checkpointed models
    # config_data['checkpoint_freq'] = -1
    # config_data['use_checkpoint'] = False # resume training from latest saved checkpoint. False = start anew

    # config_data['npcs'] = 10  # number of PCs being used; npcs should explained ~90% of the variance in mouse pose
    # config_data['max_states'] = 100 # number of maximum states (i.e., syllables) the AR-HMM will try to learn 

    # use robust AR-HMM with Student's t-distribution -> able to tolerate more noise
    config_data['robust'] = True 

    # save model object at the end of training
    # the saved model object can be used to applied to new data
    # config_data['save_model'] = True

    # use separate transition matrix for each experimental group assigned above
    # config_data['separate_trans'] = False

    # number of iterations to train model
    # Use 100-200 for finding the optimal kappa value. Use 1000 for fitting your final models
    # config_data['num_iter'] = 100 

    # syllable length probability distribution prior; (None, int or 'scan');
    # if None, kappa is set to the number of frames of your dataset
    # config_data['kappa'] = None
    if config_data.get('kappa') == 'scan':
        # config_data['scan_scale'] = 'log' # log or linear
        # config_data['min_kappa'] = None # optionally set lower bound for kappa scan
        # config_data['max_kappa'] = None # optionally set upper bound for kappa
        config_data['n_models'] = 15 # total number of models to spool
        # config_data['out_script'] = 'train_out.sh' # script name to save kappa-scanning commands


    ### Advanced Modeling Parameters ###

    # config_data['hold_out'] = False # boolean to hold out data subset during the training process
    config_data['nfolds'] = 2 # if hold_out is True, sets the number of folds to hold out during training

    # hyperparameters for hierarchical dirichlet process - leave these alone
    # config_data['alpha'] = 5.7
    # config_data['gamma'] = 1e3

    # hyperparameter for the autogregressive process - generally leave this alone too
    # config_data['nlags'] = 3

    # Select platform to run models on
    # config_data['cluster_type'] = 'local' # currently supported cluster_types = 'local' or 'slurm'
    # config_data['ncpus'] = 0 # CPUs each kappa-scan job requests on SLURM (0 requests 8); a single fit uses the CPUs the job or machine gives it

    ## SLURM PARAMETERS
    # if config_data['kappa'] == 'scan' and config_data['culuster_type'] = 'slurm'
    ## only edit these parameters if cluster_type == 'slurm'
    # config_data['prefix'] = 'source ~/.bashrc; conda activate moseq2-app; ' # prefix that activates the conda environment
    # config_data['memory'] = '16GB'
    # config_data['wall_time'] = '3:00:00'
    # config_data['partition'] = 'short'

### Train Model

Training the model can take a considerable amount of time.
Hold tight until this process finishes.

In [ ]:
from moseq2_model.gui import learn_model_command

# for a kappa scan: True runs the learn-model commands here, with progress shown in the terminal;
# False writes them to the script for you to run
learn_model_command(str(project.folder), model_name, run_cmd=True)

## Get Best Model Fit
This feature compares the trained model(s) with the principal components' changepoints and it supports two objectives for the comparison. `duration` finds the model where **the syllable duration** best matches that of the principal components' changepoints. The best match could be done on `mean`, `median` or `mode` and the default is matching the mean syllable duration with that of the changepoints. `jsd` finds the model where **the distribution of syllable durations** best match that of the principal components' changepoints. You can find more information in the [wiki](https://github.com/dattalab/moseq2-app/wiki/MoSeq2-Extract-Modeling-Notebook-Instructions#finding-best-model-fit).

A good rule of thumb is that models trained with `kappa` values within an order of magnitude of the number of frames in your dataset most closely match the model-free changepoint distribution.

Once completed, if there are multiple models, this feature returns the best model from the models in the project's `models` folder, records it in `moseq2-project.yaml` as the project's current model, and produces a `.csv` file with the summary statistics for each model saved in the same folder. The figure will be automatically saved as png and pdf in the base directory.

**Instructions:**
- **Run the following cell** to get the best model fit.

In [ ]:
from os.path import join
from IPython.display import display, Image
from moseq2_viz.gui import get_best_fit_model

output_file = join(project.folder, 'model_vs_pc_changepoints')
# objective could be either duration, jsd or median_loglikelihood
objective = 'duration (mean match)'
# change the objective to median_loglikelihood when finding the final model
# objective = 'median_loglikelihood'

best_model_fit = get_best_fit_model(str(project.folder), output_file, plot_all=True, objective=objective)

# uncomment the line below if no image is displayed
# display(Image(output_file + '.png'))

***

# Notebook End 

# User Survey

Please take some time to tell us your thoughts about this notebook:
**[user feedback survey](https://forms.gle/S88jptAEs41mQjff7)**